In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
from transformers import AutoTokenizer, TFAutoModelForTokenClassification
import json

# Check GPU availability
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("GPU Name:", tf.test.gpu_device_name())

# Set mixed precision policy
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

# Load the dataset
df = pd.read_csv('/content/train.csv')
df = df.head(100000)  # Increase this number if you have enough memory
questions1 = df['question1'].tolist()
questions2 = df['question2'].tolist()
labels = df['is_duplicate'].tolist()

num_classes = len(set(labels))

# Load AfriBERTa model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("castorini/afriberta_large")
with tf.device('/GPU:0'):
    afriberta = TFAutoModelForTokenClassification.from_pretrained("castorini/afriberta_large", from_pt=True)

# Generate embeddings for question pairs
def get_embeddings(questions1, questions2, batch_size=32):
    embeddings = []
    for i in range(0, len(questions1), batch_size):
        batch_q1 = questions1[i:i+batch_size]
        batch_q2 = questions2[i:i+batch_size]

        inputs1 = tokenizer(batch_q1, return_tensors="tf", padding=True, truncation=True, max_length=512)
        inputs2 = tokenizer(batch_q2, return_tensors="tf", padding=True, truncation=True, max_length=512)

        with tf.device('/GPU:0'):
            outputs1 = afriberta(inputs1, output_hidden_states=True)
            outputs2 = afriberta(inputs2, output_hidden_states=True)

            embeddings1 = outputs1.hidden_states[-1][:, 0, :]
            embeddings2 = outputs2.hidden_states[-1][:, 0, :]

            batch_embeddings = tf.concat([embeddings1, embeddings2], axis=-1)
            embeddings.append(batch_embeddings)

    return tf.concat(embeddings, axis=0)

dataset_embeddings = get_embeddings(questions1, questions2)

# Split the data
dataset_embeddings_np = dataset_embeddings.numpy()
X_train, X_test, y_train, y_test = train_test_split(dataset_embeddings_np, labels, test_size=0.2, random_state=42)

# LIDA Components
class Autoencoder(tf.keras.Model):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Dense(512, activation='relu'),
            tf.keras.layers.Dense(256, activation='relu'),
        ])
        self.decoder = tf.keras.Sequential([
            tf.keras.layers.Dense(512, activation='relu'),
            tf.keras.layers.Dense(input_dim, activation='linear'),
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

class DenoisingAutoEncoder(tf.keras.Model):
    def __init__(self, embedding_dim, hidden_dim, learning_rate):
        super(DenoisingAutoEncoder, self).__init__()
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_dim, activation='relu', input_shape=(embedding_dim,))
        ])
        self.decoder = tf.keras.Sequential([
            tf.keras.layers.Dense(embedding_dim, activation='linear')
        ])
        self.optimizer = tf.keras.optimizers.Adam(learning_rate)
        self.loss_fn = tf.keras.losses.MeanSquaredError()

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def add_noise(self, x, noise_frac):
        noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=0.1)
        noisy_x = x + noise_frac * noise
        return noisy_x

    def train_step(self, x, noise_frac=0.5):
        with tf.GradientTape() as tape:
            noisy_x = self.add_noise(x, noise_frac)
            reconstructions = self(noisy_x)
            loss = self.loss_fn(x, reconstructions)

        gradients = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))
        return {'loss': loss}

    def test_step(self, x):
        reconstructions = self(x)
        loss = self.loss_fn(x, reconstructions)
        return {'loss': loss}

def linear_transformation(embedding):
    return embedding + tf.random.normal(shape=tf.shape(embedding), mean=0.0, stddev=0.01)

# Train autoencoders
with tf.device('/GPU:0'):
    input_dim = X_train.shape[1]
    autoencoder = Autoencoder(input_dim)
    denoising_autoencoder = DenoisingAutoEncoder(embedding_dim=input_dim, hidden_dim=256, learning_rate=0.001)

    autoencoder.compile(optimizer='adam', loss='mse')
    autoencoder.fit(X_train, X_train, epochs=10, batch_size=32, validation_split=0.2)

    for epoch in range(10):
        for x_batch in tf.data.Dataset.from_tensor_slices(X_train).batch(32):
            denoising_autoencoder.train_step(x_batch)




Note: Runs on T4 gpu of google collab

In [ ]:
# Generate synthetic embeddings
def generate_synthetic_embeddings(original_embedding):
    linear_embed = linear_transformation(original_embedding)
    auto_embed = autoencoder(tf.expand_dims(original_embedding, axis=0))
    denoising_embed = denoising_autoencoder(tf.expand_dims(original_embedding, axis=0))

    # Cast the float16 tensors to float32 before concatenation
    auto_embed = tf.cast(auto_embed, tf.float32)
    denoising_embed = tf.cast(denoising_embed, tf.float32)

    return tf.concat([original_embedding, linear_embed, tf.squeeze(auto_embed), tf.squeeze(denoising_embed)], axis=-1)

# Use tf.data.Dataset to handle batching and mapping
synthetic_train = tf.data.Dataset.from_tensor_slices(X_train).map(generate_synthetic_embeddings).batch(32)
synthetic_test = tf.data.Dataset.from_tensor_slices(X_test).map(generate_synthetic_embeddings).batch(32)



In [ ]:
# Get input size
sample_batch = next(iter(synthetic_train))
input_size = sample_batch.shape[1]

# BiLSTM Classifier
class BiLSTMClassifier(tf.keras.Model):
    def __init__(self, input_size, hidden_size, num_classes):
        super(BiLSTMClassifier, self).__init__()
        self.lstm = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(hidden_size))
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, x):
        x = tf.expand_dims(x, axis=1)  # Add time dimension
        lstm_out = self.lstm(x)
        out = self.fc(lstm_out)
        return out

# Create and compile the classifier
with tf.device('/GPU:0'):
    hidden_size = 128
    classifier = BiLSTMClassifier(input_size, hidden_size, num_classes)
    classifier.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Convert the tf.data.Dataset to NumPy arrays, and flatten batches
    synthetic_train_np = np.concatenate([batch for batch in synthetic_train.as_numpy_iterator()], axis=0)
    y_train_np = np.array(y_train)

    classifier.fit(synthetic_train_np, y_train_np, epochs=10, batch_size=32, validation_split=0.2)



In [ ]:
# Evaluate and generate metrics
y_pred = classifier.predict(synthetic_test)
y_pred_classes = np.argmax(y_pred, axis=1)

accuracy = accuracy_score(y_test, y_pred_classes)
precision = precision_score(y_test, y_pred_classes, average='weighted')
recall = recall_score(y_test, y_pred_classes, average='weighted')
f1 = f1_score(y_test, y_pred_classes, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

